# Chronology-v2 audit and forward-score handoff

Original artifacts remain available at commit dbd03f1. This notebook verifies separately rebuilt histories and scores, including independent batch-summed reference histories. It does not open 2025 or fit a pitcher-response model. The rain buffers and isolated far-future timestamp are explicitly conservative assumptions, not validated exact completion times.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts.verify_chronology_repair import main
main()
RESULTS=ROOT/'results/chronology_repair' 

{
  "total_windows": 413081,
  "unique_games": 7289,
  "independent_reference_games": 34,
  "independent_history_equalities": 5364,
  "reloaded_training_scores": 211818,
  "reloaded_development_scores": 136487,
  "restored_game_716404_windows": 61,
  "history_invariants_passed": true,
  "score_accounting_passed": true,
  "test_season_opened": false
}
           split  common_scored  newly_scored  newly_warmup  mean_absolute_change  max_absolute_change
        training         211757            61            54              0.000017             0.060666
development_2024         136487             0             0              0.000013             0.006406


In [2]:
audit=pd.read_csv(RESULTS/'game_chronology_audit.csv')
assert len(audit)==7289 and audit.game_pk.is_unique and audit.resolved_bounds.all()
assert audit.missing_event_times.eq(0).all()
audit.groupby('season').agg(games=('game_pk','size'),resumptions=('resume_date','count'),inverted_events=('inverted_events','sum'),trailing_incomplete_plays=('trailing_incomplete_plays','sum'))

Out[2]: 
        games  resumptions  inverted_events  trailing_incomplete_plays
season                                                                
2022     2430            1                0                          2
2023     2430            7                9                          0
2024     2429            3                2                          0


,games,resumptions,inverted_events,trailing_incomplete_plays
season,,,,
2022,2430,1,0,2
2023,2430,7,9,0
2024,2429,3,2,0


In [3]:
pd.read_csv(RESULTS/'history_change_summary.csv')

Out[3]: 
   season       n  changed_history_rows  max_abs_opportunity_change
0    2022  137732                   495                          19
1    2023  138862                   337                          20
2    2024  136487                   421                         164


,season,n,changed_history_rows,max_abs_opportunity_change
0,2022,137732,495,19
1,2023,138862,337,20
2,2024,136487,421,164


In [4]:
pd.read_csv(RESULTS/'prediction_changes.csv')

Out[4]: 
              split  common_scored  ...  mean_absolute_change  max_absolute_change
0          training         211757  ...              0.000017             0.060666
1  development_2024         136487  ...              0.000013             0.006406

[2 rows x 6 columns]


,split,common_scored,newly_scored,newly_warmup,mean_absolute_change,max_absolute_change
0,training,211757,61,54,0.000017,0.060666
1,development_2024,136487,0,0,0.000013,0.006406


In [5]:
pd.read_csv(ROOT/'results/forward_threat_v2/metrics.csv')

Out[5]: 
              split                   model  ...     brier  average_precision
0      forward_2022                     raw  ...  0.012653           0.066083
1      forward_2022  prospective_calibrated  ...  0.012625           0.065101
2      forward_2023                     raw  ...  0.016475           0.081500
3      forward_2023  prospective_calibrated  ...  0.016400           0.081003
4    selection_2023                     raw  ...  0.015936           0.088959
5    selection_2023  prospective_calibrated  ...  0.015858           0.088677
6  development_2024                     raw  ...  0.017221           0.087252
7  development_2024  prospective_calibrated  ...  0.017128           0.087252

[8 rows x 11 columns]


,split,model,n,attempts,mean_prediction,observed_rate,calibration_intercept,calibration_slope,log_loss,brier,average_precision
0,forward_2022,raw,72956,953,0.012708,0.013063,-0.171729,0.946317,0.060173,0.012653,0.066083
1,forward_2022,prospective_calibrated,72956,953,0.012601,0.013063,0.040961,1.000817,0.060192,0.012625,0.065101
2,forward_2023,raw,138862,2377,0.016612,0.017118,-0.274119,0.908993,0.073931,0.016475,0.081500
3,forward_2023,prospective_calibrated,138862,2377,0.017631,0.017118,-0.058129,0.992224,0.073878,0.016400,0.081003
4,selection_2023,raw,68864,1150,0.017431,0.016700,-0.268943,0.931621,0.071355,0.015936,0.088959
5,selection_2023,prospective_calibrated,68864,1150,0.018297,0.016700,0.018262,1.035372,0.071402,0.015858,0.088677
6,development_2024,raw,136487,2462,0.017716,0.018038,-0.409567,0.867754,0.076797,0.017221,0.087252
7,development_2024,prospective_calibrated,136487,2462,0.018014,0.018038,-0.179700,0.945929,0.076571,0.017128,0.087252


In [6]:
json.loads((ROOT/'results/forward_threat_v2/policy.json').read_text())

Out[6]: 
{'selected_policy': 'identity',
 'selection_raw': {'log_loss': 0.07135533855495531,
  'brier': 0.01593620139623858,
  'average_precision': 0.08895877028396804},
 'selection_calibrated': {'log_loss': 0.07140211478357392,
  'brier': 0.01585778737294823,
  'average_precision': 0.08867658399119058},
 'final_calibration_n': 138862,
 'final_calibration_attempts': 2377,
 'final_intercept': -0.27411917615902276,
 'final_slope': 0.9089931407445644}


{'selected_policy': 'identity',
 'selection_raw': {'log_loss': 0.07135533855495531,
  'brier': 0.01593620139623858,
  'average_precision': 0.08895877028396804},
 'selection_calibrated': {'log_loss': 0.07140211478357392,
  'brier': 0.01585778737294823,
  'average_precision': 0.08867658399119058},
 'final_calibration_n': 138862,
 'final_calibration_attempts': 2377,
 'final_intercept': -0.27411917615902276,
 'final_slope': 0.9089931407445644}

The training-only selection remains identity calibration. The next-stage response protocol is frozen separately; this audit does not establish causal effects or real-world action completeness. See docs/chronology_repair_report.md and docs/pitcher_response_protocol.md.